In [17]:
from langchain.agents import create_agent
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore, LocalFileStore, create_kv_docstore
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import Chroma
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.documents import Document
from langchain_core.embeddings import FakeEmbeddings
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field

# TYPE TETRIS 2 

| | investiga | grafo | uso |
|---|---|---|---|
| `probe(obj)` | contratos de tipo (assinaturas, Unions) | grafo de **tipos** — cadeia sequencial | `probe=ChatOpenAI().invoke` `probe=ChatOpenAI` |
| `target(obj)` | um objeto que já existe | grafo de **instâncias** | `target=ChatOpenAI()` `target=ChatOpenAI().invoke()`

> Rule of Thumb: classes/instances → target
>> functions/callables (including these create_* factories) → probe.
>>> probe reads contracts, target reads the public surface, siblings searches sideways — but nothing reads what an object is actually holding. 

In [2]:
# working packages — tooling dependencies for probe/target/siblings/_align_left,
# kept separate from cell 0 (which holds only the LangChain targets being inspected)
import html
import importlib
import inspect
import pkgutil
import typing

import pandas as pd
from IPython.display import HTML, Markdown, display


In [3]:
# %%catch — wraps a cell's execution so an exception can never halt "Run All".
from IPython.core.magic import register_cell_magic


@register_cell_magic
def catch(line, cell):
    result = get_ipython().run_cell(cell)
    if not result.success:
        print("⚠️ cell failed — continuing to the next cell")

## Helper visual

`_align_left` só formata o DataFrame pra ficar alinhado à esquerda (mesmo
truque do notebook original). Usado por `probe` e por `target`, sem log
nenhum por trás.

In [4]:
def _align_left(df: pd.DataFrame):
    styler = (
        df.style.format(escape="html")
        .set_properties(**{"text-align": "left"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
    )
    if "required" in df.columns:
        styler = styler.apply(
            lambda row: ["color: orange" if row["required"] else "" for _ in row],
            axis=1,
        )
    # privado = fora do contrato público; cinza pra nunca confundir com o resto.
    if "private" in df.columns:
        styler = styler.apply(
            lambda row: ["color: #888888" if row["private"] else "" for _ in row],
            axis=1,
        )
    # hop = o valor é um objeto de outra classe; dá pra continuar o tetris nele.
    if "hop" in df.columns:
        styler = styler.apply(
            lambda row: ["color: #4ec9b0" if row["hop"] else "" for _ in row],
            axis=1,
        )
    return styler

## `probe(obj, label=None)`

Igual antes na deteção do que `obj` é (typing construct / callable / raw
object), mas agora **não substitui** o output — empilha. Cada chamada
guarda o resultado num log (`_probe_log`, até 5 entradas, FIFO) e redesenha
o log inteiro embaixo da célula, com listras claro/escuro alternando pra
separar visualmente uma pesquisa da outra. Pra limpar tudo, chame
`clear_probe_all()`; pra apagar só uma entrada específica (pela posição
mostrada em `#N/total`), chame `clear_probe_1()` .. `clear_probe_5()`.

Probe is for contracts of things you call, not things you instantiate.

In [5]:
_probe_log = []  # até 5 entradas, mais recente por último


def clear_probe_all():
    _probe_log.clear()
    _render_probe_log()


def _clear_probe_entry(position):
    if 1 <= position <= len(_probe_log):
        del _probe_log[position - 1]
    _render_probe_log()


clear_probe_1 = lambda: _clear_probe_entry(1)
clear_probe_2 = lambda: _clear_probe_entry(2)
clear_probe_3 = lambda: _clear_probe_entry(3)
clear_probe_4 = lambda: _clear_probe_entry(4)
clear_probe_5 = lambda: _clear_probe_entry(5)


def _render_probe_log():
    total = len(_probe_log)
    style = (
        "<style>.tetris-log-entry table thead th {"
        "padding-top:8px; padding-bottom:8px; border-bottom:2px solid #3c3f44;}"
        "</style>"
    )
    blocks = []
    for i, entry in enumerate(_probe_log):
        bg = "#2a2d34" if i % 2 == 0 else "#1e2126"
        blocks.append(
            f'<div class="tetris-log-entry" style="background:{bg}; color:#d4d4d4; padding:10px 14px; '
            f'border-bottom:1px solid #3c3f44;">'
            f'<div style="font-weight:bold; color:#9cdcfe; font-size:15px; padding-top:8px; padding-bottom:8px;">'
            f"#{i + 1}/{total}&nbsp;&nbsp;&nbsp;{html.escape(entry['short_name'])}"
            f"</div>"
            f'<div style="font-weight:normal; font-size:12px; color:#ffffff; margin-bottom:6px;">'
            f"{html.escape(entry['label'])} \u2014 "
            f'<span style="font-weight:normal; font-style:italic; color:#9cdcfe;">{html.escape(entry["kind"])}</span>'
            f"</div>{entry['body']}</div>"
        )
    display(HTML(style + "".join(blocks)))


def probe(obj, label: str | None = None):
    label = label or getattr(obj, "__name__", repr(obj))
    short_name = getattr(obj, "__name__", None) or type(obj).__name__
    args = typing.get_args(obj)

    if args:
        origin = typing.get_origin(obj)
        kind = f"typing construct ({getattr(origin, '__name__', repr(origin))})"
        df = pd.DataFrame(
            [
                {"index": i, "arg": repr(a), "type_of_arg": type(a).__name__}
                for i, a in enumerate(args)
            ]
        )
        body = f"<pre>tipo completo:\n{html.escape(repr(obj))}</pre>" + (
            _align_left(df).to_html() if not df.empty else ""
        )
    elif callable(obj):
        kind = "callable signature"
        sig = inspect.signature(obj)
        df = pd.DataFrame(
            [
                {
                    "name": name,
                    "kind": p.kind.name,
                    "default": repr(p.default),
                    "required": p.default is inspect.Parameter.empty
                    and p.kind
                    not in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD),
                    "annotation": str(p.annotation),
                }
                for name, p in sig.parameters.items()
            ]
        )
        body = _align_left(df).to_html() if not df.empty else "<i>(sem par\u00e2metros)</i>"
    else:
        kind = "raw object"
        df = pd.DataFrame([{"repr": repr(obj), "type": type(obj).__name__}])
        body = _align_left(df).to_html()

    _probe_log.append({"label": label, "kind": kind, "short_name": short_name, "body": body})
    del _probe_log[:-5]
    _render_probe_log()

## `target(obj, label=None, private=False)`

Mesma detecção de sempre (abstração / construção / dados / métodos), só
que os 4 blocos agora vão para dentro de **uma entrada** do log
(`_target_log`, até 5, FIFO), e cada chamada redesenha o log inteiro com
listras claro/escuro alternando por entrada. Pra limpar tudo, chame
`clear_target_all()`; pra apagar só uma entrada específica (pela posição
mostrada em `#N/total`), chame `clear_target_1()` .. `clear_target_5()`.

### `private=False` — o filtro que esconde a máquina

Por padrão `target` pula todo nome que começa com `_`: é o **contrato
público**, o que a lib promete manter. Mas em classes-wrapper (quase toda
a LangChain) o objeto de verdade mora justamente atrás desse underscore —
`Chroma._collection` é o `chromadb.Collection` real, e não existe nenhum
equivalente público (`Chroma` não expõe *nada* que conte documentos).

`private=True` inclui os nomes de um underscore só; dunders (`__x__`)
continuam sempre fora, porque são ruído. As linhas privadas saem em cinza,
e a coluna `private` marca cada uma — o contrato público continua legível
de relance.

Regra: **investigue com `private=True`, dependa só do que aparece com
`private=False`.** O que está em cinza pode sumir na próxima release.

In [6]:
def _describe_valor(value):
    r = repr(value)
    if len(r) <= 60:
        return r
    if isinstance(value, str):
        return f"str ({len(value)} chars)"
    if isinstance(value, dict):
        return f"dict ({len(value)} keys)"
    if isinstance(value, (list, tuple, set)):
        return f"{type(value).__name__} ({len(value)} items)"
    return f"{type(value).__name__} (repr longo: {len(r)} chars)"


def _hidden(name: str, private: bool) -> bool:
    """Dunder nunca aparece; um underscore só aparece quando private=True."""
    if name.startswith("__"):
        return True
    return name.startswith("_") and not private


_target_log = []  # até 5 entradas, mais recente por último


def clear_target_all():
    _target_log.clear()
    _render_target_log()


def _clear_target_entry(position):
    if 1 <= position <= len(_target_log):
        del _target_log[position - 1]
    _render_target_log()


clear_target_1 = lambda: _clear_target_entry(1)
clear_target_2 = lambda: _clear_target_entry(2)
clear_target_3 = lambda: _clear_target_entry(3)
clear_target_4 = lambda: _clear_target_entry(4)
clear_target_5 = lambda: _clear_target_entry(5)


def _render_target_log():
    total = len(_target_log)
    style = (
        "<style>.tetris-log-entry table thead th {"
        "padding-top:8px; padding-bottom:8px; border-bottom:2px solid #3c3f44;}"
        "</style>"
    )
    blocks = []
    for i, entry in enumerate(_target_log):
        bg = "#2a2d34" if i % 2 == 0 else "#1e2126"
        blocks.append(
            f'<div class="tetris-log-entry" style="background:{bg}; color:#d4d4d4; padding:10px 14px; '
            f'border-bottom:1px solid #3c3f44;">'
            f'<div style="font-weight:bold; color:#9cdcfe; font-size:15px; padding-top:8px; padding-bottom:8px;">'
            f"#{i + 1}/{total}&nbsp;&nbsp;&nbsp;{html.escape(entry['short_name'])}"
            f"</div>"
            f'<div style="font-weight:normal; font-size:12px; color:#ffffff; margin-bottom:6px;">'
            f"{html.escape(entry['label'])}"
            f"</div>{entry['body']}</div>"
        )
    display(HTML(style + "".join(blocks)))


def target(obj, label: str | None = None, private: bool = False):
    cls = obj if inspect.isclass(obj) else type(obj)
    label = label or cls.__name__
    parts = []

    # abstração
    is_abstract = inspect.isabstract(cls)
    pending = sorted(cls.__abstractmethods__) if is_abstract else []
    status = (
        f"abstrata — métodos pendentes: {', '.join(pending)}"
        if is_abstract
        else "concreta (instanciável)"
    )
    parts.append(f"<p><b>abstração:</b> {status}</p>")

    # escopo — deixa explícito na tabela se o que está à vista é o contrato
    # público ou a máquina inteira, pra nunca ler uma coisa achando que é a outra
    escopo = "público + privado (_)" if private else "só público (private=True mostra o resto)"
    parts.append(f"<p><b>escopo:</b> {escopo}</p>")

    # construção
    sig = inspect.signature(cls)
    df_construcao = pd.DataFrame(
        [
            {
                "name": name,
                "kind": p.kind.name,
                "default": repr(p.default),
                "required": p.default is inspect.Parameter.empty
                and p.kind not in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD),
                "annotation": str(p.annotation),
            }
            for name, p in sig.parameters.items()
        ]
    )
    parts.append(
        "<p><b>construção:</b></p>"
        + (
            _align_left(df_construcao).to_html()
            if not df_construcao.empty
            else "<i>(sem parâmetros)</i>"
        )
    )

    # dados
    rows = []
    for name in dir(obj):
        if _hidden(name, private):
            continue
        try:
            value = getattr(obj, name)
        except Exception:
            continue
        if callable(value):
            continue
        rows.append(
            {
                "name": name,
                "private": name.startswith("_"),
                "resumo": _describe_valor(value),
                "print[:40]": str(value)[:40],
            }
        )
    df_dados = pd.DataFrame(rows, columns=["name", "private", "resumo", "print[:40]"])
    if not df_dados.empty:
        df_dados = df_dados.sort_values("name").reset_index(drop=True)
    parts.append(
        "<p><b>dados da instância:</b></p>"
        + (_align_left(df_dados).to_html() if not df_dados.empty else "<i>(nenhum)</i>")
    )

    # métodos
    rows = []
    for name in dir(obj):
        if _hidden(name, private):
            continue
        try:
            value = getattr(obj, name)
        except Exception:
            continue
        if not callable(value):
            continue
        self_repr = type(value.__self__).__name__ if hasattr(value, "__self__") else "?"
        is_pending = name in cls.__abstractmethods__ if is_abstract else False
        rows.append(
            {
                "name": name,
                "private": name.startswith("_"),
                "method": f"{self_repr}.{name}",
                "abstract": is_pending,
            }
        )
    df_metodos = pd.DataFrame(rows, columns=["name", "private", "method", "abstract"])
    if not df_metodos.empty:
        df_metodos = df_metodos.sort_values("name").reset_index(drop=True)
    parts.append(
        "<p><b>métodos da instância:</b></p>"
        + (_align_left(df_metodos).to_html() if not df_metodos.empty else "<i>(nenhum)</i>")
    )

    body = "".join(parts)
    _target_log.append({"label": label, "short_name": cls.__name__, "body": body})
    del _target_log[:-5]
    _render_target_log()

## `state(obj, label=None)`

A lente que faltava. `dir(obj)` mostra a **superfície da classe** — tudo que
dá pra chamar, herdado ou não. `vars(obj)` mostra o **estado da instância** —
só o que o `__init__` de fato guardou *neste* objeto. Em wrapper (quase toda
a LangChain) os dois divergem muito:

| | responde | em `Chroma` |
|---|---|---|
| `target` / `dir` | o que dá pra chamar | 39 nomes públicos |
| `full_init_params` | o que dá pra **passar** | params do `__init__` via MRO |
| `state` / `vars` | o que ele **guardou** | 6 nomes — e o motor está aí |

A coluna `hop` (em verde) é o ponto: marca todo valor que é objeto de outra
classe, ou seja, onde o tetris **continua**. `Chroma` guarda um
`chromadb.Collection` em `_collection` — `hop=True` — e é nele que mora o
`.count()` que a `Chroma` nunca expôs.

### classe ≠ instância

`vars` responde coisas diferentes conforme o que você passa, e a linha
**nível** diz qual das duas você está vendo:

- `state(Chroma)` → **namespace da classe**: métodos e atributos de classe.
- `state(Chroma.from_texts(...))` → **estado da instância**: o que aquele
  objeto guardou. É aqui que `_collection` aparece.

Passar a classe quando você queria a instância é o erro fácil — a tabela vem
cheia e mesmo assim sem o motor. Dunders (`__x__`) ficam sempre fora dos dois.

Objetos com `__slots__` (ou builtins) não têm `__dict__`; nesse caso `state`
avisa e você volta pro `target(..., private=True)`.

### log e limpeza

Igual `probe` e `target`: cada chamada empilha uma entrada em `_state_log`
(até 5, FIFO) e redesenha o log inteiro com listras alternadas. Pra limpar
tudo, `clear_state_all()`; pra apagar só uma entrada (pela posição mostrada
em `#N/total`), `clear_state_1()` .. `clear_state_5()`.

In [7]:
def _is_hop(value) -> bool:
    """O valor é um objeto de outra classe — vale continuar o tetris nele?"""
    if value is None or isinstance(value, (str, bytes, bool, int, float, complex)):
        return False
    if isinstance(value, (list, tuple, set, frozenset, dict)):
        return False
    return type(value).__module__ != "builtins"


_state_log = []  # até 5 entradas, mais recente por último


def clear_state_all():
    _state_log.clear()
    _render_state_log()


def _clear_state_entry(position):
    if 1 <= position <= len(_state_log):
        del _state_log[position - 1]
    _render_state_log()


clear_state_1 = lambda: _clear_state_entry(1)
clear_state_2 = lambda: _clear_state_entry(2)
clear_state_3 = lambda: _clear_state_entry(3)
clear_state_4 = lambda: _clear_state_entry(4)
clear_state_5 = lambda: _clear_state_entry(5)


def _render_state_log():
    total = len(_state_log)
    style = (
        "<style>.tetris-log-entry table thead th {"
        "padding-top:8px; padding-bottom:8px; border-bottom:2px solid #3c3f44;}"
        "</style>"
    )
    blocks = []
    for i, entry in enumerate(_state_log):
        bg = "#2a2d34" if i % 2 == 0 else "#1e2126"
        blocks.append(
            f'<div class="tetris-log-entry" style="background:{bg}; color:#d4d4d4; padding:10px 14px; '
            f'border-bottom:1px solid #3c3f44;">'
            f'<div style="font-weight:bold; color:#9cdcfe; font-size:15px; padding-top:8px; padding-bottom:8px;">'
            f"#{i + 1}/{total}&nbsp;&nbsp;&nbsp;{html.escape(entry['short_name'])}"
            f"</div>"
            f'<div style="font-weight:normal; font-size:12px; color:#ffffff; margin-bottom:6px;">'
            f"{html.escape(entry['label'])}"
            f"</div>{entry['body']}</div>"
        )
    display(HTML(style + "".join(blocks)))


def state(obj, label: str | None = None):
    label = label or getattr(obj, "__name__", None) or type(obj).__name__
    short_name = getattr(obj, "__name__", None) or type(obj).__name__
    parts = []

    # nível — `vars` responde coisas diferentes pra classe e pra instância, e
    # confundir os dois é ler o namespace da classe achando que é estado do objeto
    is_class = inspect.isclass(obj)
    nivel = (
        "classe — namespace da própria classe (`vars(cls)`): métodos e atributos de classe"
        if is_class
        else "instância — o que o `__init__` guardou neste objeto (`vars(obj)`)"
    )
    parts.append(f"<p><b>nível:</b> {nivel}</p>")

    holdings = vars(obj) if hasattr(obj, "__dict__") else {}
    # dunder sempre fora (ruído); um underscore só é justamente o que interessa aqui
    holdings = {k: v for k, v in holdings.items() if not _hidden(k, private=True)}

    if not holdings:
        parts.append(
            "<p><i>(sem `__dict__` utilizável — objeto com `__slots__` ou builtin. "
            "Use `target(obj, private=True)`.)</i></p>"
        )
    else:
        rows = [
            {
                "name": name,
                "type": type(value).__name__,
                "hop": _is_hop(value),
                "from": type(value).__module__,
                "resumo": _describe_valor(value),
            }
            for name, value in holdings.items()
        ]
        df = pd.DataFrame(rows, columns=["name", "type", "hop", "from", "resumo"])
        df = df.sort_values("name").reset_index(drop=True)
        parts.append("<p><b>guardado:</b></p>" + _align_left(df).to_html())

        hops = [r["name"] for r in rows if r["hop"]]
        if hops:
            alvo = short_name if is_class else "obj"
            sugestao = " &middot; ".join(
                f"<code>target({alvo}.{h}, private=True)</code>" for h in hops
            )
            parts.append(f"<p><b>próximo hop:</b> {sugestao}</p>")

    body = "".join(parts)
    _state_log.append({"label": label, "short_name": short_name, "body": body})
    del _state_log[:-5]
    _render_state_log()

## `full_init_params(cls)`

Resolve o buraco negro do `**kwargs`: nem `probe` nem `target` conseguem ver o
que uma subclasse repassa pro `__init__` da classe-mãe via
`super().__init__(**kwargs)` — a assinatura da subclasse só mostra
`**kwargs: Any`, sem dizer o que cabe lá dentro.

`full_init_params` percorre `inspect.getmro(cls)` (a cadeia de herança) e,
pra cada ancestral que define seu **próprio** `__init__` (não herdado — via
`"__init__" in vars(klass)`), extrai os parâmetros reais desse `__init__`.
Resultado: uma tabela única com todo parâmetro que a classe aceita, direta
ou indiretamente, junto com o nome da classe de onde ele realmente vem
(coluna `from`).

Use quando `probe(cls)` ou a tabela de "construção" do `target(cls)`
terminarem em `**kwargs: Any` e você precisar saber o que está escondido
ali.

In [8]:
def full_init_params(cls):
    seen = {}
    for klass in inspect.getmro(cls):
        if "__init__" not in vars(klass):  # só __init__ definido NESSA classe, não herdado
            continue
        for name, p in inspect.signature(klass.__init__).parameters.items():
            if name in ("self", "args", "kwargs"):
                continue
            seen.setdefault(
                name, (klass.__name__, p)
            )  # primeira ocorrência = mais específica na MRO

    df = pd.DataFrame(
        [
            {
                "name": name,
                "from": owner,
                "kind": p.kind.name,
                "default": repr(p.default),
                "required": p.default is inspect.Parameter.empty
                and p.kind not in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD),
                "annotation": str(p.annotation),
            }
            for name, (owner, p) in seen.items()
        ]
    )
    display(Markdown(f"**{cls.__name__}** — _full init params (via MRO)_"))
    display(_align_left(df) if not df.empty else df)

### `**kwargs` em métodos — quando `full_init_params` não basta

1. Método (classmethod ou de instância) também aceita `**kwargs` — e é até mais comum que em função solta (ex: `from_documents`/`from_texts` do LangChain).
2. Diferente do caso de classe, aqui não tem MRO: o método só repassa `**kwargs` pra dentro do próprio corpo — uma chamada arbitrária, não uma cadeia formal.
3. Padrão "factory" (`from_x`) costuma desembocar em `cls(**kwargs)`, isto é, no `__init__` da própria classe.
4. Nesse caso, `full_init_params(cls)` ainda serve — só que aplicado na classe, não no método.
5. Não é garantido: hops intermediários (ex: `from_texts` dentro de `from_documents`) podem extrair parâmetros nomeados que não aparecem nem na assinatura do método nem em `full_init_params`.

**Fluxo rápido:**
1. `probe(classe.metodo)` — vê a assinatura explícita + o `**kwargs` opaco.
2. Se suspeitar de um factory que desemboca no construtor: `full_init_params(classe)` — vê o que o `__init__` final aceita.
3. Se tiver hop intermediário no meio: `inspect.getsource(classe.metodo)` — único jeito de confirmar automaticamente o que cada hop extrai.

## `siblings(cls, label=None)`

Resolve "quem implementa isso?" — o que nem `probe` nem `target` fazem
sozinhos, porque os dois só investigam o objeto que você já aponta pra
eles; nenhum faz busca reversa tipo "liste as subclasses de `X` que
existem no pacote".

Sobe da classe pro pacote-pai (via `cls.__module__`, cortando o último
segmento) e cobre dois modos de busca ali dentro:

- **Modo A — `__all__`**: a lista que o autor do pacote declarou como API
  pública. Confiável quando existe, mas nem todo pacote a mantém completa
  — ex: `langchain.chains.combine_documents` só exporta funções auxiliares
  em `__all__`, nunca as classes de chain (`StuffDocumentsChain` etc.).
- **Modo B — `pkgutil.iter_modules` + `issubclass`**: ignora `__all__` e lê
  a estrutura real de arquivos do pacote. Importa cada submódulo-irmão e
  filtra só as classes que (a) foram **definidas** ali — não reimportadas
  de outro lugar — e (b) são de fato subclasse de `cls`. Sempre funciona,
  mesmo quando o Modo A não ajuda em nada.

**Limitação:** os dois modos partem do pressuposto de que a classe-base
mora dentro de um **subpacote dedicado** (uma pasta com vários `.py`, um
por implementação — como `combine_documents/`). Quando a classe-base é só
um módulo solto dentro de um pacote gigante (ex: `BaseStore`, que vive em
`langchain_core/stores.py`, dentro do enorme `langchain_core`), `siblings`
sobe pro pacote errado — grande demais, sem sinal útil. Nesse caso, ainda
vale a pista manual (import já existente em algum notebook, doc oficial,
etc.) — ver S2 abaixo.

In [9]:
def siblings(cls, label: str | None = None):
    label = label or cls.__name__
    leaf_mod_name = cls.__module__
    pkg_name = leaf_mod_name.rsplit(".", 1)[0] if "." in leaf_mod_name else leaf_mod_name
    pkg = importlib.import_module(pkg_name)

    display(Markdown(f"**{label}** — _pacote: `{pkg.__name__}`_"))

    # Modo A — __all__ (API pública declarada pelo autor do pacote)
    display(Markdown("Modo A — `__all__`:"))
    exported = getattr(pkg, "__all__", None)
    df_a = pd.DataFrame({"name": sorted(exported)}) if exported else pd.DataFrame(columns=["name"])
    display(_align_left(df_a) if not df_a.empty else df_a)

    # Modo B — módulos irmãos (estrutura real do pacote) + filtro por issubclass
    display(Markdown(f"Modo B — subclasses reais de `{label}` nos módulos irmãos:"))
    rows = []
    if hasattr(pkg, "__path__"):
        for modinfo in pkgutil.iter_modules(pkg.__path__):
            try:
                submod = importlib.import_module(f"{pkg.__name__}.{modinfo.name}")
            except Exception:
                continue
            for name, obj in inspect.getmembers(submod, inspect.isclass):
                if obj.__module__ != submod.__name__:
                    continue  # só classes definidas NESSE módulo, não reimportadas
                if obj is cls or not issubclass(obj, cls):
                    continue
                rows.append({"module": modinfo.name, "class": name})
    df_b = pd.DataFrame(rows, columns=["module", "class"])
    if not df_b.empty:
        df_b = df_b.sort_values(["module", "class"]).reset_index(drop=True)
    display(_align_left(df_b) if not df_b.empty else df_b)

# `PROBE`

In [10]:
vector_store = Chroma(
    collection_name="tetris_sandbox",
    embedding_function=FakeEmbeddings(size=8),
)

/var/folders/pf/7lwsqjw92g96dl5sfdckf9_r0000gn/T/ipykernel_1602/3815223000.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [11]:
_ = vector_store._collection.count

probe(_, label=str(_))

# clear_probe_all()  # descomente e rode pra zerar o log inteiro
# clear_probe_1()    # ou apague só a entrada #1 (troque o número)

# `TARGET`

In [19]:
_ = ConversationalRetrievalChain


target(_, label=str(_))

# clear_target_all()  # descomente e rode pra zerar o log inteiro
# clear_target_1()  # ou apague só a entrada #1 (troque o número)

<!-- # `KWARGS` -->

# STATE

In [13]:
_ = vector_store


state(_, label=str(_))

# clear_state_all()  # descomente e rode pra zerar o log inteiro
# clear_state_1()  # ou apague só a entrada #1 (troque o número)

,name,type,hop,from,resumo
0,_client,Client,True,chromadb.api.client,<chromadb.api.client.Client object at 0x116479ac0>
1,_client_settings,Settings,True,chromadb.config,Settings (repr longo: 2876 chars)
2,_collection,Collection,True,chromadb.api.models.Collection,Collection(name=tetris_sandbox)
3,_embedding_function,FakeEmbeddings,True,langchain_core.embeddings.fake,FakeEmbeddings(size=8)
4,_persist_directory,str,False,builtins,'./chroma'
5,override_relevance_score_fn,NoneType,False,builtins,None


# FULL_INIT_PARAMS

In [14]:
# full_init_params quer a CLASSE crua (sem parênteses, sem .method) — a
# tabela é 100% estática (assinaturas de __init__ via MRO), não depende de
# nenhuma instância. CharacterTextSplitter() ou CharacterTextSplitter.metodo
# dão AttributeError (não tem __mro__) e, mesmo funcionando, não mudariam a
# tabela — então nem vale normalizar isso na função.

_ = CharacterTextSplitter

full_init_params(_)

# clear_full_init_params_all()  # descomente e rode pra zerar o log inteiro
# clear_full_init_params_1()  # ou apague só a entrada #1 (troque o número)

**CharacterTextSplitter** — _full init params (via MRO)_

,name,from,kind,default,required,annotation
0,separator,CharacterTextSplitter,POSITIONAL_OR_KEYWORD,'\n\n',False,str
1,is_separator_regex,CharacterTextSplitter,POSITIONAL_OR_KEYWORD,False,False,bool
2,chunk_size,TextSplitter,POSITIONAL_OR_KEYWORD,4000,False,int
3,chunk_overlap,TextSplitter,POSITIONAL_OR_KEYWORD,200,False,int
4,length_function,TextSplitter,POSITIONAL_OR_KEYWORD,<built-in function len>,False,"Callable[[str], int]"
5,keep_separator,TextSplitter,POSITIONAL_OR_KEYWORD,False,False,"bool | Literal['start', 'end']"
6,add_start_index,TextSplitter,POSITIONAL_OR_KEYWORD,False,False,bool
7,strip_whitespace,TextSplitter,POSITIONAL_OR_KEYWORD,True,False,bool


# `SIBLINGS`

In [15]:
_ = BaseMessage

siblings(_, label=str(_))

NameError: name 'BaseMessage' is not defined

## S2 — a limitação na prática

`BaseStore` vive num módulo solto (`langchain_core/stores.py`), não num
subpacote dedicado. `siblings` sobe pro pacote inteiro `langchain_core` —
grande demais pra ser útil. Serve como lembrete: quando isso acontecer,
volte pra pista manual (import já existente, doc oficial).

In [ ]:
_ = Chroma

siblings(_, label=str(_))

**<class 'langchain_community.vectorstores.chroma.Chroma'>** — _pacote: `langchain_community.vectorstores`_

Modo A — `__all__`:

,name
0,Aerospike
1,AlibabaCloudOpenSearch
2,AlibabaCloudOpenSearchSettings
3,AnalyticDB
4,Annoy
5,ApacheDoris
6,ApertureDB
7,AstraDB
8,AtlasDB
9,AwaDB


Modo B — subclasses reais de `<class 'langchain_community.vectorstores.chroma.Chroma'>` nos módulos irmãos:

,module,class


In [ ]:
# the tetris 2 is a initiative to develop skill to handle new packages without having to know it by heart. i am studying RAG, agents, so the number of new packages ahead poses a challenge. So the overall goal is to instead of give the fish, teach how to fishing.
# One of my main strategies is to start from the class i want to know more and dig deep into its constructor signature cycling probe -> target and all the tetris 2 features.

# right now i am dealing with a subject called conversation buffer, specifically the classes ConversationBufferMemory and ConversationChain

reescrevendo: vamos entender o processo parte por parte. comecando com o dados, a pagina web vira chunks pai e filho. Os chunks filhos sao embedados e guardados no chroma em formato vetorial. Eles contem metadados com os IDs apontando para os chunks pai, que estao em formato texto na memoria indexados por Id.

o retriever embeda uma query e a compara no chroma. os vetores mais similares tem seus IDs recuperados na memoria. Desta forma, os pedacos menores de texto que foram embedados e guardados no Chrome, servem para dar fidelidade ao serem transformados em vetor e serem comparados com outros vetores, como neste caso a query do usuario. Esse vetores similares possuem ID que apontam para trechos maiores, os chunks pai, presentes na memoria em formato texto e indexados por ID. O retriever entao retorna estes trechos pai para llm como contexto. portanto a query do usuario é usada para primeiro resgatar os trechos adequados e munir o llm do texto necessario para responder à query. Depois o llm, de posse deste contexto, cria a resposta para a query. 